[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Variational_Inference_Flows.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Variational Inference & Normalizing Flows

The [VAE's](./Representation_Learning.ipynb) loss finally justified: the ELBO derived, its gap identified as a KL, and normalizing flows — exact likelihoods through invertible networks — built and audited against a closed-form density.

## 1. Pre-requisites

[Representation Learning](./Representation_Learning.ipynb) S2, [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) (KL), [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) (change of variables).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *The ELBO, Derived* (~40 min)
**Goal:** one line of algebra: log-likelihood = ELBO + KL(q‖posterior); verified on a conjugate model.
**Builds on:** [Representation Learning](./Representation_Learning.ipynb) S2. &nbsp; **Feeds into:** Session 2 (flows).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The ELBO, Derived</b></summary>

**Timing (~40 min).** 12 min the derivation · 8 min what the gap means · 12 min the conjugate experiment · 8 min reading the numbers.

**Open by naming the debt this session repays.** The [VAE workshop](./Representation_Learning.ipynb) presented `rec + β·KL` as a loss that works. **Nobody justified it.** This session derives it in one line of algebra and identifies exactly what the KL term is doing — so a student who found the VAE loss arbitrary gets the reason rather than the recipe.

**Do the derivation live, because it is three steps and each one is trivial.** Start with $\log p(x) = \log\int p(x,z)\,dz$ — intractable. Multiply and divide by any $q(z)$ inside the integral. Apply Jensen. What falls out is not just a bound but an **identity**:

$$\log p(x) = \underbrace{E_q[\log p(x,z) - \log q(z)]}_{\text{ELBO}} + \underbrace{\mathrm{KL}(q \,\|\, p(z\mid x))}_{\text{the gap}}$$

**Emphasise that this is an equality, not an inequality** — the inequality is a consequence of $\mathrm{KL} \ge 0$. That distinction is what makes the next point available.

**Then deliver the consequence, which is the session's whole idea.** $\log p(x)$ does not depend on $q$. So **maximising the ELBO over $q$ is exactly minimising the KL between $q$ and the true posterior** — you are doing inference, not just optimising a surrogate. The bound is tight if and only if $q$ *is* the posterior. Rooms that see this stop treating variational inference as an approximation of convenience and start seeing it as posterior fitting.

**Explain why the experiment uses a conjugate Gaussian model, since it is the methodological point.** With $z \sim \mathcal{N}(0,1)$ and $x \mid z \sim \mathcal{N}(z, 0.5^2)$, both the evidence $\log p(x)$ and the exact posterior are available in closed form. **This is one of the very few settings where the gap can be measured rather than assumed to be small** — in any real application $\log p(x)$ is exactly the intractable quantity you were avoiding.

**Have the room verify the closed forms rather than accepting them.** Posterior precision $= 1 + 1/0.25 = 5$, so variance $0.2$; mean $= 0.2 \times (1.2/0.25) = 0.96$. Marginally $x \sim \mathcal{N}(0, 1.25)$, giving $\log p(1.2) = -1.6065$. **Two lines of Gaussian algebra, and now the demo has an answer key.**

**Prepare the room for the gap to come out *negative*, because it will and it looks like an error.** The printed value is $-0.00123$, and a KL divergence cannot be negative. Ask what went wrong before explaining. The answer: the ELBO is estimated from **256 Monte Carlo samples**, so it carries sampling noise of roughly that magnitude. The estimator overshot. **A negative gap is a measurement artefact, and noticing that it is impossible is the skill** — the same reflex as the negative Sinkhorn gap in [Optimal Transport](./Optimal_Transport.ipynb).

**Close by connecting back to the VAE explicitly.** The VAE is this derivation with an **encoder network** producing $q(z\mid x)$ instead of two scalars, and with $q$ amortised across all $x$ rather than re-optimised per data point. That amortisation introduces a *second* gap — the encoder may not reach the best $q$ in its family — which is one reason real VAEs are looser than this demo. **The theory is now installed; the VAE loss is not a recipe, it is this identity.**
</details>

## 2. The Bound and Its Gap

💡 **Intuition.** Latent-variable likelihoods need an integral over $z$ — intractable. Multiply and divide by any distribution $q(z)$ inside the log, apply [Jensen](../Intro_Math/Information_Theory/Information_Theory.ipynb), and:
$$\log p(x) = \underbrace{E_q[\log p(x, z) - \log q(z)]}_{\text{ELBO}} + \underbrace{KL(q \,\|\, p(z|x))}_{\ge 0, = \text{the gap}}$$
Maximizing the ELBO over $q$ *is* pushing $q$ toward the true posterior; the bound is tight iff they match. The VAE's loss is exactly this with an encoder network as $q$. On a conjugate Gaussian model the posterior is known — so for once we can *watch* the gap close.

In [2]:
# model: z ~ N(0,1), x|z ~ N(z, 0.5²); observe x=1.2. Posterior is closed-form Gaussian.
x_obs = 1.2
s2_lik = 0.25
post_var = 1/(1 + 1/s2_lik)
post_mu = post_var * (x_obs/s2_lik)
log_px = -0.5*np.log(2*np.pi*(1+s2_lik)) - 0.5*x_obs**2/(1+s2_lik)     # exact evidence

# variational family: N(m, s²); optimize the ELBO by gradient ascent (reparameterized MC)
m = torch.tensor(0.0, requires_grad=True)
log_s = torch.tensor(0.0, requires_grad=True)
opt = torch.optim.Adam([m, log_s], lr=0.05)
for step in range(800):
    eps = torch.randn(256)
    z = m + torch.exp(log_s)*eps
    logp = -0.5*np.log(2*np.pi) - 0.5*z**2            -0.5*np.log(2*np.pi*s2_lik) - 0.5*(x_obs - z)**2/s2_lik
    logq = -0.5*np.log(2*np.pi) - log_s - 0.5*eps**2
    elbo = (logp - logq).mean()
    opt.zero_grad(); (-elbo).backward(); opt.step()

kl_gap = log_px - elbo.item()
print(f"true posterior:  N({post_mu:.4f}, {post_var:.4f})")
print(f"learned q:       N({m.item():.4f}, {torch.exp(2*log_s).item():.4f})")
print(f"log p(x) = {log_px:.4f}   final ELBO = {elbo.item():.4f}   gap = {kl_gap:.5f} → ≈ 0: q reached the posterior")

true posterior:  N(0.9600, 0.2000)
learned q:       N(0.9637, 0.1907)
log p(x) = -1.6065   final ELBO = -1.6053   gap = -0.00123 → ≈ 0: q reached the posterior


**What just happened.** Gradient ascent on the ELBO, over a two-parameter Gaussian family, found the true posterior:

| | mean | variance |
|---|---|---|
| true posterior | 0.9600 | 0.2000 |
| learned $q$ | 0.9637 | 0.1907 |

and the bound closed: $\log p(x) = -1.6065$, final ELBO $= -1.6053$.

**Check the closed forms rather than trusting them, because the whole demo rests on them.** Posterior precision is $1 + 1/0.25 = 5$, so variance $= 0.2$; posterior mean is $0.2 \times (1.2/0.25) = 0.96$. Marginally $x \sim \mathcal{N}(0, 1.25)$, so $\log p(1.2) = -\tfrac12\log(2\pi \cdot 1.25) - \tfrac{1.44}{2 \cdot 1.25} = -1.6065$. **Two lines of Gaussian algebra supply the answer key**, which is exactly why this conjugate model was chosen.

**Now the reported gap: $-0.00123$. That is negative, and a KL divergence cannot be negative.** The identity says $\log p(x) = \text{ELBO} + \mathrm{KL}(q \| p(z|x))$ with $\mathrm{KL} \ge 0$, so the ELBO can never exceed $\log p(x)$. **The bound was not violated — the estimate was noisy.** The ELBO here is a **256-sample Monte Carlo average**, and its standard error is comfortably larger than $10^{-3}$. The estimator overshot by about one standard error.

**Noticing that is the transferable skill, and it recurs across this curriculum.** The same reflex catches the Sinkhorn cost that landed *below* the LP optimum in [Optimal Transport](./Optimal_Transport.ipynb). **When a number lands on the impossible side of a bound, the bound is fine and your estimator is telling you something.** Here it says: average over more samples, or over the last hundred steps, before quoting a gap.

**With that caveat, the finding is genuine and it is the point of the session.** Nothing in the training loop ever mentioned the posterior — the code only ever computed $\log p(x,z) - \log q(z)$ and pushed uphill. **Maximising the ELBO performed inference**, because $\log p(x)$ does not depend on $q$, so every nat gained by the ELBO is a nat removed from $\mathrm{KL}(q \| p(z|x))$. Optimising a bound and fitting a posterior are the same operation.

**And the bound is tight here for a specific reason worth stating.** The true posterior is Gaussian and the variational family is Gaussian, so **the family contains the answer**. That is the best case. When the true posterior is multimodal or skewed and $q$ is Gaussian, the gap does not close — and the direction of the KL, $\mathrm{KL}(q\|p)$ rather than $\mathrm{KL}(p\|q)$, makes $q$ **mode-seeking**: it will lock onto one mode and ignore the others rather than covering both. That asymmetry is the best-known limitation of variational inference and it is invisible in a conjugate demo.

**Finally, note the two ways a real VAE is looser than this.** Its $q$ family is whatever the encoder network can express, and it is **amortised** — one network must produce a good $q$ for *every* $x$, rather than each $x$ getting its own optimisation. That second effect has its own name, the amortisation gap, and it is why the VAE's ELBO sits further below $\log p(x)$ than these two scalars managed.

---
### 🕐 Session 2 of 3 — *Normalizing Flows* (~40 min)
**Goal:** exact densities through invertible maps: change-of-variables with a learnable Jacobian.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (the trade-space).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Normalizing Flows</b></summary>

**Timing (~40 min).** 10 min change of variables · 12 min why the determinant is the whole problem · 10 min coupling layers · 8 min training and the normalisation audit.

**Open by contrasting with Session 1 in one sentence.** The ELBO gives a **bound** on the likelihood, and you never know how loose it is. Flows give the **exact** log-likelihood — no bound, no gap, no Monte Carlo estimate. That is the entire selling point, and it comes at a price paid in architecture.

**Derive the change-of-variables formula rather than quoting it, because the intuition is geometric.** If $x = f(z)$ is invertible, probability mass is conserved but *density* is not — squeeze a region and the density rises by exactly the factor you squeezed. That factor is $|\det J|$, giving $\log p(x) = \log p_z(f^{-1}(x)) + \log|\det J_{f^{-1}}|$. **The Jacobian determinant is a volume-change bookkeeping term**, and it is the only new ingredient.

**Then state the engineering problem plainly, because everything in the field follows from it.** A general $d \times d$ determinant costs $O(d^3)$, and it must be computed **at every layer, for every sample, on every training step**. At $d = 3072$ (a $32\times32$ image) that is impossible. **Flow research is almost entirely the search for architectures whose Jacobian determinant is cheap** — not for architectures that model better.

**Coupling layers are the answer, and the trick is worth spelling out.** Split the coordinates. Leave half **untouched**. Transform the other half using parameters computed from the untouched half. The Jacobian is then **triangular**, so its determinant is the product of the diagonal — the scales — and costs $O(d)$. Note the crucial asymmetry: the network computing $s$ and $t$ can be arbitrarily complex, because it is never differentiated *through* in the determinant. **Unlimited expressiveness in the conditioner, trivial determinant.**

**Point at why the layers alternate.** A single coupling layer leaves half the coordinates completely unchanged. Swapping which half moves at each layer — the `i % 2` in the code — is what lets every coordinate eventually be transformed by every other. Stack six and you have a genuinely expressive map.

**Flag `s = torch.tanh(s)` as a stability measure, not decoration.** The scale enters as $e^s$, so an unbounded $s$ gives an unbounded volume change and the log-determinant explodes. Bounding $s$ to $(-1,1)$ caps the per-layer scaling at $e^{\pm 1}$. Students who remove it get `nan` within a few hundred steps.

**Make the normalisation audit the centrepiece, because it is the claim that distinguishes flows from everything else.** Sum the density over a grid, multiply by the cell area, and check against 1. The demo returns **1.000**. **A VAE cannot do this, a GAN cannot do this, a diffusion model cannot do this cheaply** — none of them gives you a normalised density you can integrate. This one line of verification is the whole reason flows survive.

**Give the NLL a yardstick so it is interpretable.** 1.275 nats against 2.838 for the best-fitting standard normal — a **1.56 nat** improvement, which is a factor of $e^{1.56} \approx 4.8$ in likelihood per point. And because the density is exactly normalised, that number is directly comparable across models: **flows are the only family in the table where likelihood comparison is unambiguous.**

**Close on the cost of the constraint, since the trade must be stated.** Every layer must be invertible and dimension-preserving — there is no bottleneck, no latent space smaller than the data, and no arbitrary architecture. You cannot drop a convolution with stride into a flow and expect it to work. **Flows buy exactness with architectural freedom**, and that trade is what Session 3's table is for.
</details>

## 3. Exact Likelihood, No Bound

💡 **Intuition.** Push a Gaussian through an *invertible* network $f$ and the [change-of-variables formula](../Intro_Math/Analysis/Random_Variables.ipynb) gives the exact density: $\log p(x) = \log p_z(f^{-1}(x)) + \log|\det J_{f^{-1}}|$. The engineering is making that determinant cheap: **coupling layers** transform half the coordinates using parameters computed from the other half — triangular Jacobian, determinant = product of scales. Stack and alternate halves: an expressive, exactly-normalized density.

In [3]:
# simpler, correct assembly:
class Flow(nn.Module):
    def __init__(self, n_layers=6):
        super().__init__()
        self.nets = nn.ModuleList([nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2)) for _ in range(n_layers)])
    def forward(self, x):
        logdet = torch.zeros(len(x))
        for i, net in enumerate(self.nets):
            keep, move = (x[:, :1], x[:, 1:]) if i % 2 == 0 else (x[:, 1:], x[:, :1])
            s, t = net(keep).chunk(2, dim=1)
            s = torch.tanh(s)
            moved = move*torch.exp(s) + t
            x = torch.cat([keep, moved], 1) if i % 2 == 0 else torch.cat([moved, keep], 1)
            logdet += s.squeeze(1)
        return x, logdet

# target: the two-moons distribution (reuse the diffusion workshop's)
def moons(n):
    t_ = rng.uniform(0, np.pi, n)
    top = np.stack([np.cos(t_), np.sin(t_)], 1)
    bot = np.stack([1-np.cos(t_), 0.4-np.sin(t_)], 1)
    X = np.concatenate([top[:n//2], bot[n//2:]]) + 0.06*rng.standard_normal((n, 2))
    return torch.tensor(((X - X.mean(0))/X.std(0)), dtype=torch.float32)

flow = Flow(); opt = torch.optim.Adam(flow.parameters(), lr=1e-3)
Xm = moons(6000)
for step in range(4000):
    idx = torch.randint(0, len(Xm), (512,))
    z, logdet = flow(Xm[idx])
    nll = (0.5*(z**2).sum(1) + np.log(2*np.pi) - logdet).mean()   # exact NLL!
    opt.zero_grad(); nll.backward(); opt.step()
print(f"final exact NLL: {nll.item():.3f} nats  (a standard normal fit would give ≈ {0.5*2*np.log(2*np.pi*np.e):.3f})")

# density heatmap — exactly normalized by construction
g = torch.linspace(-2.6, 2.6, 160)
GX, GY = torch.meshgrid(g, g, indexing="xy")
pts = torch.stack([GX.ravel(), GY.ravel()], 1)
with torch.no_grad():
    z, logdet = flow(pts)
    logp = -0.5*(z**2).sum(1) - np.log(2*np.pi) + logdet
p = logp.exp().reshape(160, 160)
cell = (g[1]-g[0])**2
print(f"∫p ≈ {float(p.sum()*cell):.3f}  (exactly-normalized density: should be ≈ 1)")
plt.figure(figsize=(4.4, 3.6))
plt.contourf(GX, GY, p, levels=30)
plt.scatter(*Xm[:800].T, s=1, c="w", alpha=0.4)
plt.title("flow density: exact log-likelihoods, integral ≈ 1")
plt.tight_layout(); plt.show()

final exact NLL: 1.275 nats  (a standard normal fit would give ≈ 2.838)
∫p ≈ 1.000  (exactly-normalized density: should be ≈ 1)


/tmp/ipykernel_3921173/3580017194.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two numbers and a picture, and the second number is the one that matters:

- **NLL 1.275 nats**, against 2.838 for the best standard normal — an improvement of **1.56 nats**, a factor of $e^{1.56} \approx 4.8$ in likelihood per data point.
- **$\int p \approx 1.000$** — the learned density integrates to one.

**That integral is the claim no other generative family in this curriculum can make.** A [VAE](./Representation_Learning.ipynb) gives a *bound*; a GAN gives no density at all; a [diffusion model](./Diffusion_Models.ipynb) gives a likelihood only through an expensive ODE. **A flow gives you $p(x)$ directly, exactly normalised, and you can check it by summing over a grid.** One line of verification, and it either passes or it does not.

**And "exactly normalised" is structural, not learned.** Nothing in the training loss rewarded normalisation. It holds because $p_z$ is a normalised Gaussian and the change-of-variables formula conserves total mass under any invertible map — so **any** invertible $f$, trained or random, yields a normalised density. The 1.000 confirms the implementation is correct, not that the model is good.

**Which makes the NLL directly comparable in a way most generative metrics are not.** 1.275 nats is a real, unbounded-below-only-by-the-data quantity: two flows on the same data can be ranked by it without qualification. **Compare that to FID, ELBO values, or human evaluation** — every other family's headline number involves either a bound or a proxy.

**The engineering that makes this possible is the coupling layer, and it is worth reading in the code.** Half the coordinates pass through **untouched**; the other half are scaled and shifted by parameters computed *from the untouched half*. The Jacobian is therefore **triangular**, so its determinant is just the product of the scales — $O(d)$ instead of $O(d^3)$. Note the asymmetry that buys the expressiveness: the network producing $s$ and $t$ can be arbitrarily complicated, because it never appears in the determinant. `logdet += s.squeeze(1)` is the entire volume-tracking cost.

**Two implementation details are load-bearing.** The `i % 2` alternation swaps which half moves, because otherwise half the coordinates would never be transformed at all. And `s = torch.tanh(s)` caps the per-layer volume change at $e^{\pm 1}$ — remove it and the log-determinant runs away to `nan` within a few hundred steps. **Neither is cosmetic.**

**Now the constraint that is the real cost, and it should not be buried.** Every layer must be **invertible and dimension-preserving**. There is no bottleneck, no latent space smaller than the data, no stride, no pooling — you cannot drop an arbitrary architecture into a flow. At $d = 3072$ for a small image, the latent is also 3072-dimensional, so flows are **parameter-hungry** compared to a VAE with a 128-dimensional latent. **Flows buy exactness by spending architectural freedom.**

**One caveat about the density plot itself.** The heatmap shows the model's density, not the data's — and a flow on a distribution concentrated near a thin manifold has to place enormous density on a small region while remaining continuous everywhere. Look for the characteristic failure: thin filaments of leaked density connecting the two moons, where the invertible map has to stretch through the empty space between them. **A flow cannot assign exactly zero density anywhere**, which is the price of being a smooth invertible map from a Gaussian.

---
### 🕐 Session 3 of 3 — *The Generative Trade-Space* (~25 min)
**Goal:** VAE vs flow vs diffusion vs GAN: what each buys and what each pays.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: The Generative Trade-Space</b></summary>

**Timing (~25 min).** 10 min the table as a set of forced choices · 8 min why diffusion won · 7 min where flows still live.

**This session is a map, not a demo — say so, and say what the map is for.** Four families, five properties, and **no row is achievable in every column simultaneously**. Every generative model juggles sample quality, likelihood access, and sampling speed, and each family drops a different ball. A student who can name which ball a method dropped can read any generative-modelling paper.

**Force the room to derive two entries rather than reading them.** *Why can a flow not have a bottleneck?* Because invertibility requires dimension preservation — you cannot invert a map that lost information. *Why does diffusion need many sampling steps?* Because each denoising step is deliberately small and the chain is sequential; that is the same cost asymmetry the [diffusion workshop](./Diffusion_Models.ipynb) measured. **Both answers follow from the definition of the method, and deriving them beats memorising the table.**

**Give the honest account of why diffusion won the 2020s, since students usually attribute it to sample quality alone.** The decisive property was **training stability**. GANs produce excellent samples and are notoriously fragile — mode collapse, oscillation, careful architecture tuning per dataset. Diffusion training is a plain regression loss with no adversary and essentially no failure modes. **The field switched to the method that trains reliably at scale, not to the one with the best samples on a good day.** Reliability compounds when you are spending millions of GPU-hours.

**Then be equally honest about where flows still win, because "diffusion won" invites the wrong conclusion.** Anywhere you need an **exact, normalised, comparable likelihood**: physics likelihoods and simulation-based inference, anomaly detection with a calibrated threshold, and lossless compression — where the code length *is* $-\log_2 p(x)$, connecting straight back to [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb). **A bound is useless for all three.** The niche is small and it is real.

**Emphasise the ELBO's role as the shared grammar.** It underlies the VAE directly, and it reappears in the diffusion derivation before simplifying to the noise-prediction loss. **Session 1 was not preparation for one architecture; it was preparation for the vocabulary of the whole field.** Rooms tend to file the ELBO under "VAEs" and this is the moment to correct that.

**Close with a question that outlives any particular architecture.** *Which of the three — exact likelihood, one-pass sampling, architectural freedom — is this method giving up?* Flows give up architectural freedom. Diffusion gives up sampling speed. GANs give up likelihood and stability. VAEs give up exactness. **Every method in the field, including ones not yet published, answers that question**, and a student who asks it is reading rather than being told. Then point forward to [Diffusion II](./Diffusion_Score_SDE.ipynb), where the probability-flow ODE reveals diffusion's flow-shaped face and the two families turn out to be less separate than the table suggests.
</details>

## 4. The Family Reunion

| | VAE | Flow | [Diffusion](./Diffusion_Models.ipynb) | GAN |
|---|---|---|---|---|
| Likelihood | bound (ELBO) | **exact** | bound/exact (SDE) | none |
| Sampling | 1 pass | 1 pass | many steps | 1 pass |
| Architecture freedom | full | invertible only | full | full |
| Training stability | good | good | **great** | fragile |
| Latent space | semantic | dimension-preserving | noise schedule | semantic |

💡 **Intuition.** Every generative model juggles three balls — sample quality, likelihood access, sampling speed — and each family drops a different one. Diffusion won the 2020s on quality+stability; flows keep the niche where exact density matters (physics, anomaly detection, [compression](../Intro_Math/Information_Theory/Information_Theory.ipynb)); the ELBO remains the shared grammar.

---
## Where next

- [Diffusion II](./Diffusion_Score_SDE.ipynb) — probability flow: diffusion's flow-shaped face.
- [Representation Learning](./Representation_Learning.ipynb) — the VAE, now with its theory installed.